# OcupaPerú — Fase 3 CRISP-DM: Preparación de los datos

**Problema:** predecir si un perfil hotelero *(departamento × clase × categoría × mes)* tendrá **alta ocupación**.

**Fuente:** MINCETUR (DGIETA) — *Indicadores de Ocupabilidad* (datosabiertos.gob.pe, licencia Open Data Commons Attribution: uso libre citando la fuente) · 11 archivos anuales 2015 → 2025-S1.

Este notebook **construye el dataset**: unifica dos esquemas incompatibles, audita las 8 dimensiones de calidad, limpia, **elimina las variables con fuga de información** y genera las features.

| Etapa | Qué hace |
|---|---|
| 1 | Carga y normalización de esquema (**schema drift**) |
| 2 | Auditoría de calidad — 8 dimensiones |
| 3 | Limpieza (agregados, imposibles, ceros no informativos) |
| 4 | **Detección y exclusión de data leakage** |
| 5 | Ingeniería de características |
| 6 | Split temporal y exportación |

> 📂 Sube los 11 archivos `Indicadores_ocupabilidad_*.csv` a **`/content`** (panel izquierdo 📁 → *Subir al almacenamiento de sesión*) antes de correr.

In [ ]:
# Colab: los CSV deben estar en /content
%cd /content
print('📂 Sube los 11 archivos Indicadores_ocupabilidad_*.csv a /content')
print('   (panel izquierdo 📁 → Subir al almacenamiento de sesión) y vuelve a ejecutar.')

In [ ]:
import pandas as pd, numpy as np, glob, warnings
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings('ignore')

PALETA = ['#8B1A1A', '#C8A951', '#2E4057', '#5B8C5A', '#A64B2A']
sns.set_theme(style='whitegrid'); plt.rcParams['figure.dpi'] = 110
pd.set_option('display.width', 160)

print('Entorno listo')

---
## Etapa 1 — Carga y normalización de esquema

**La trampa:** MINCETUR cambió *todos* los nombres de columna en 2024.

| Años | Esquema |
|---|---|
| 2015–2023 | `AÑO`, `DESCRIPCIÓN CLASE`, `TNOH EN EL MES(%)` — 23 columnas |
| 2024–2025 | `ANIO`, `CLASE`, `PORCENTAJE_TNOH`, `FECHA_CORTE` — 24 columnas |

Un `pd.concat()` sin normalizar **no lanza error**: devuelve 46 columnas medio vacías y sigue corriendo. Falla en silencio.

Además el separador decimal cambia: **coma** hasta 2024, **punto** en 2025.

In [ ]:
MAPA_ESQUEMA_VIEJO = {
    'AÑO':'ANIO', 'MES':'MES', 'CLASE':'ID_CLASE', 'DESCRIPCIÓN CLASE':'CLASE',
    'CATEGORÍA':'ID_CATEGORIA', 'DESCRIPCIÓN CATEGORÍA':'CATEGORIA',
    'DEPARTAMENTO':'DEPARTAMENTO', 'UBIGEO':'ID_UBIGEO',
    'NÚMERO DE ESTABLECIMIENTO':'NUMERO_ESTABLECIMIENTOS',
    'NÚMERO DE HABITACIONES':'NUMERO_HABITACIONES',
    'NÚMERO DE PLAZAS-CAMA':'NUMERO_PLAZAS_CAMA',
    'TNOH EN EL MES(%)':'PORCENTAJE_TNOH', 'TNOC EN EL MES(%)':'PORCENTAJE_TNOC',
    'PROM PERMANENCIA(DÍAS)':'PROMEDIO_PERMANENCIA',
    'PROM PERMANENCIA - NAC (DÍAS)':'PROMEDIO_PERMANENCIA_NAC',
    'PROM PERMANENCIA - EXT (DÍAS)':'PROMEDIO_PERMANENCIA_EXT',
    'TOTAL DE ARRIBOS EN EL MES':'TOTAL_ARRIBOS',
    'TOTAL DE ARRIBOS MES - NAC':'TOTAL_ARRIBOS_NAC',
    'TOTAL DE ARRIBOS MES - EXT':'TOTAL_ARRIBOS_EXT',
    'TOTAL PERNOCT MES':'TOTAL_PERNOCT',
    'TOTAL PERNOCT MES - NAC (DÍAS)':'TOTAL_PERNOCT_NAC',
    'TOTAL PERNOCT MES - EXT (DÍAS)':'TOTAL_PERNOCT_EXT',
    'TOTAL EMPLEO EN EL MES':'TOTAL_EMPLEO',
}

COLS_NUM = ['NUMERO_ESTABLECIMIENTOS','NUMERO_HABITACIONES','NUMERO_PLAZAS_CAMA',
    'PORCENTAJE_TNOH','PORCENTAJE_TNOC','PROMEDIO_PERMANENCIA','PROMEDIO_PERMANENCIA_NAC',
    'PROMEDIO_PERMANENCIA_EXT','TOTAL_ARRIBOS','TOTAL_ARRIBOS_NAC','TOTAL_ARRIBOS_EXT',
    'TOTAL_PERNOCT','TOTAL_PERNOCT_NAC','TOTAL_PERNOCT_EXT','TOTAL_EMPLEO']

archivos = sorted(glob.glob('Indicadores_ocupabilidad_*.csv'))
if len(archivos) != 11:
    raise FileNotFoundError(f'Se esperaban 11 archivos Indicadores_ocupabilidad_*.csv y hay {len(archivos)}. Súbelos a /content.')
frames, bitacora = [], []

for f in archivos:
    d = pd.read_csv(f, sep=';', encoding='latin-1', dtype=str)
    n_cols = d.shape[1]
    if 'FECHA_CORTE' in d.columns:
        esquema = 'NUEVO'
        d = d.drop(columns=['FECHA_CORTE'])
    else:
        esquema = 'VIEJO'
        d = d.rename(columns=MAPA_ESQUEMA_VIEJO)
    d['ARCHIVO_ORIGEN'] = f
    bitacora.append({'archivo': f, 'esquema': esquema,
                     'cols_orig': n_cols, 'filas': len(d)})
    frames.append(d)

bitacora = pd.DataFrame(bitacora)
print(bitacora.to_string(index=False))
print(f"\n>>> Esquemas distintos detectados: {bitacora.esquema.nunique()}   <-- LA TRAMPA")

crudo = pd.concat(frames, ignore_index=True)
print(f">>> Crudo unificado: {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")

In [ ]:
# Normalización de decimales: coma (2015-2024) -> punto (2025)
for c in COLS_NUM:
    crudo[c] = pd.to_numeric(
        crudo[c].astype(str).str.strip().str.replace(',', '.', regex=False),
        errors='coerce')

crudo['ANIO'] = crudo['ANIO'].astype(int)
crudo['MES']  = crudo['MES'].astype(int)
crudo['FECHA'] = pd.to_datetime(dict(year=crudo.ANIO, month=crudo.MES, day=1))
crudo['DEPARTAMENTO'] = crudo['DEPARTAMENTO'].str.strip().str.upper()

print("Nulos tras conversión numérica:", int(crudo[COLS_NUM].isna().sum().sum()))
print("Periodo:", crudo.FECHA.min().date(), "→", crudo.FECHA.max().date(),
      f"({crudo.FECHA.nunique()} meses)")
crudo.head(3)

---
## Etapa 2 — Auditoría de calidad (8 dimensiones)

In [ ]:
LLAVE = ['ANIO','MES','DEPARTAMENTO','CLASE','CATEGORIA']

es_agregado    = crudo.CLASE.eq('TODAS CONSOLIDADAS') | crudo.CATEGORIA.eq('TODAS CONSOLIDADAS')
tnoh_imposible = (crudo.PORCENTAJE_TNOH < 0) | (crudo.PORCENTAJE_TNOH > 100)
no_reporto     = (crudo.PORCENTAJE_TNOH == 0) & (crudo.TOTAL_ARRIBOS == 0) & (crudo.TOTAL_EMPLEO == 0)
series_ok      = (crudo.groupby('DEPARTAMENTO').FECHA.nunique() == crudo.FECHA.nunique()).sum()

auditoria = pd.DataFrame([
 ('1. Completitud',  '% de celdas numéricas no nulas',    f"{(1-crudo[COLS_NUM].isna().mean().mean())*100:.2f}%", 'OK'),
 ('2. Unicidad',     'duplicados en la llave primaria',    f"{crudo.duplicated(LLAVE).sum()}",                     'OK'),
 ('3. Consistencia', 'esquemas de columnas distintos',     f"{bitacora.esquema.nunique()}",                        'PROBLEMA'),
 ('4. Validez',      'TNOH fuera del rango [0, 100]',      f"{tnoh_imposible.sum()}",                              'PROBLEMA'),
 ('5. Exactitud',    'ceros simultáneos (= no reportó)',   f"{no_reporto.sum()}",                                  'PROBLEMA'),
 ('6. Integridad',   'filas agregadas TODAS CONSOLIDADAS', f"{es_agregado.sum()}",                                 'PROBLEMA'),
 ('7. Puntualidad',  'meses cubiertos (consecutivos)',     f"{crudo.FECHA.nunique()}",                             'OK'),
 ('8. Cobertura',    'departamentos con serie completa',   f"{series_ok} de {crudo.DEPARTAMENTO.nunique()}",        'OK'),
], columns=['Dimensión','Chequeo','Resultado','Estado'])

display(auditoria)

In [ ]:
# Los valores imposibles NO son aleatorios: son sistemáticos
imp = crudo[tnoh_imposible][['ANIO','MES','DEPARTAMENTO','CLASE','CATEGORIA',
                             'PORCENTAJE_TNOH','NUMERO_ESTABLECIMIENTOS']]
print(f"TNOH imposible: {len(imp)} filas  |  máximo = {imp.PORCENTAJE_TNOH.max():.2f}%")
print("\nDistribución por departamento y año:")
print(imp.groupby(['DEPARTAMENTO','ANIO']).size().to_string())
print("\n>>> No es un typo suelto: es Callao, hoteles de 2 estrellas, mes tras mes de 2015.")
imp.head(6)

In [ ]:
# El cráter COVID: el indicador de ocupación MINTIÓ, el de arribos no
det = crudo[~es_agregado]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))

s1 = det.groupby('FECHA').apply(lambda x: np.average(x.PORCENTAJE_TNOH.fillna(0),
                                                     weights=x.NUMERO_HABITACIONES))
ax[0].plot(s1.index, s1.values, color=PALETA[0], lw=1.6)
ax[0].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'), alpha=.15, color='grey')
ax[0].set_title('TNOH nacional (%) — baja a 15%, NO a 0'); ax[0].set_ylim(0, None)

s2 = det.groupby('FECHA').TOTAL_ARRIBOS.sum() / 1e6
ax[1].plot(s2.index, s2.values, color=PALETA[2], lw=1.6)
ax[1].axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2021-12-01'), alpha=.15, color='grey')
ax[1].set_title('Arribos totales (millones) — SÍ colapsa'); ax[1].set_ylim(0, None)
plt.tight_layout(); plt.show()

print("Abril 2019  vs  Abril 2020:")
for col, lbl in [('TOTAL_ARRIBOS','Arribos totales'), ('TOTAL_ARRIBOS_EXT','Arribos extranjeros')]:
    a = det[(det.ANIO==2019) & (det.MES==4)][col].sum()
    b = det[(det.ANIO==2020) & (det.MES==4)][col].sum()
    print(f"  {lbl:22s} {a:>11,.0f} → {b:>9,.0f}   ({(b/a-1)*100:+.1f}%)")

print("\n>>> SESGO DE SUPERVIVENCIA: la encuesta solo recoge hospedajes que reportan.")
print(">>> Los que cerraron salen del denominador, así que el TNOH no refleja el colapso real.")

---
## Etapa 3 — Limpieza

| # | Acción | Por qué |
|---|---|---|
| L1 | Eliminar filas `TODAS CONSOLIDADAS` | Son **agregados**: cada hotel se contaría 2–3 veces |
| L2 | TNOH fuera de [0,100] → `NaN` | Físicamente imposible (había uno de **244.89%**) |
| L3 | Ceros simultáneos → `NaN` + flag `NO_REPORTO` | "No reportó" ≠ "hotel vacío" |

In [ ]:
df = crudo.copy()
n0 = len(df)

# L1 — fuera los agregados
df = df[~(df.CLASE.eq('TODAS CONSOLIDADAS') | df.CATEGORIA.eq('TODAS CONSOLIDADAS'))].copy()
print(f"[L1] agregados eliminados : {n0:,} → {len(df):,}")

# L2 — imposibles
m = (df.PORCENTAJE_TNOH < 0) | (df.PORCENTAJE_TNOH > 100)
df.loc[m, COLS_NUM] = np.nan
print(f"[L2] TNOH imposible → NaN : {m.sum()} filas")

# L3 — ceros no informativos
m = (df.PORCENTAJE_TNOH == 0) & (df.TOTAL_ARRIBOS == 0) & (df.TOTAL_EMPLEO == 0)
df['NO_REPORTO'] = m.astype(int)
df.loc[m, COLS_NUM] = np.nan
print(f"[L3] no reportó → NaN     : {m.sum():,} filas ({m.mean()*100:.1f}%)")

df = df.dropna(subset=['PORCENTAJE_TNOH']).copy()
limpio = df.copy()          # <-- conserva los 126 meses: insumo de la serie mensual

print(f"\n>>> LIMPIO: {len(df):,} filas | {df.FECHA.nunique()} meses | {df.DEPARTAMENTO.nunique()} departamentos")

---
## Etapa 4 — 🚨 Data leakage

El dataset trae variables que **son el target disfrazado**. Un modelo entrenado con ellas da ROC-AUC ≈ 0.99 y **no sirve para nada**: para usarlo habría que conocer las pernoctaciones del mes… que es justo lo que se quiere predecir.

El TNOH se calcula como `pernoctaciones / (habitaciones × días del mes)`. **Todo lo que entra en esa fórmula, sale.**

In [ ]:
# Prueba 1 — reconstruir el target desde sus propios insumos
dias = df.MES.map({1:31,2:28,3:31,4:30,5:31,6:30,7:31,8:31,9:30,10:31,11:30,12:31})
tnoh_reconstruido = 100 * df.TOTAL_PERNOCT / (df.NUMERO_HABITACIONES * dias)
print(f"Correlación TNOH real vs reconstruido (PERNOCT/HABITACIONES):  r = {df.PORCENTAJE_TNOH.corr(tnoh_reconstruido):.3f}")

# Prueba 2 — correlación de cada candidata con el target
CANDIDATAS = ['PORCENTAJE_TNOC','TOTAL_PERNOCT','TOTAL_PERNOCT_NAC','TOTAL_PERNOCT_EXT',
              'TOTAL_ARRIBOS','TOTAL_ARRIBOS_NAC','TOTAL_ARRIBOS_EXT','TOTAL_EMPLEO',
              'PROMEDIO_PERMANENCIA','PROMEDIO_PERMANENCIA_NAC','PROMEDIO_PERMANENCIA_EXT',
              'NUMERO_ESTABLECIMIENTOS','NUMERO_HABITACIONES','NUMERO_PLAZAS_CAMA']
corr = df[CANDIDATAS + ['PORCENTAJE_TNOH']].corr()['PORCENTAJE_TNOH'].drop('PORCENTAJE_TNOH')

fig, ax = plt.subplots(figsize=(7.5, 4.8))
c = corr.sort_values()
ax.barh(c.index, c.values, color=['#8B1A1A' if abs(v) > 0.5 else '#9aa5b1' for v in c.values])
ax.axvline(0.5, ls='--', c='#8B1A1A', lw=1); ax.axvline(-0.5, ls='--', c='#8B1A1A', lw=1)
ax.set_title('Correlación con el target — en rojo, LEAKAGE')
plt.tight_layout(); plt.show()

print(corr.sort_values(key=abs, ascending=False).round(3).to_string())

In [ ]:
# Variables EXCLUIDAS por fuga de información
LEAKAGE = [
 'PORCENTAJE_TNOC',                                        # r ≈ 0.93 — métrica gemela (ocupación de camas)
 'TOTAL_PERNOCT','TOTAL_PERNOCT_NAC','TOTAL_PERNOCT_EXT',  # numerador del TNOH
 'TOTAL_ARRIBOS','TOTAL_ARRIBOS_NAC','TOTAL_ARRIBOS_EXT',  # resultado del mes, no insumo
 'TOTAL_EMPLEO',                                           # se ajusta a la demanda observada
 'PROMEDIO_PERMANENCIA','PROMEDIO_PERMANENCIA_NAC','PROMEDIO_PERMANENCIA_EXT',
]

print(f"EXCLUIDAS ({len(LEAKAGE)}):")
for v in LEAKAGE:
    print("  ✗", v)
print("\nREGLA: el modelo solo usa lo que se conoce ANTES de que el mes empiece —")
print("       geografía, tipo de establecimiento, calendario y REZAGOS históricos.")

---
## Etapa 5 — Ingeniería de características

**Target:** `PORCENTAJE_TNOH` binarizado en el percentil 75 → `Y_ALTA_OCUPACION`.

| Grupo | Variables |
|---|---|
| Calendario | mes (cíclico sin/cos), trimestre, fiestas patrias, semana santa, temporada seca, COVID, tendencia |
| **Rezagos** | TNOH en t-1, t-3, t-12; medias móviles 3m y 12m; delta interanual |
| Estructura | nº establecimientos, habitaciones, hab/estab, plazas/hab, nº de estrellas |
| Perfil histórico | % extranjeros, TNOH y estacionalidad del departamento — **calculados solo con ≤ 2019** |

> Los **rezagos son legítimos**: son datos *pasados*, conocidos antes del mes. Las pernoctaciones del mes en curso, no.
> El perfil histórico usa solo 2015–2019 para que no filtre información del test (2024–2025).

In [ ]:
df = df.sort_values(['DEPARTAMENTO','CLASE','CATEGORIA','FECHA']).reset_index(drop=True)

# ---------- TARGET ----------
P75 = df.PORCENTAJE_TNOH.quantile(0.75)
df['Y_ALTA_OCUPACION'] = (df.PORCENTAJE_TNOH > P75).astype(int)
print(f"Umbral p75 = {P75:.2f}%   |   clase positiva = {df.Y_ALTA_OCUPACION.mean()*100:.1f}%")

fig, ax = plt.subplots(figsize=(7.5, 3.2))
ax.hist(df.PORCENTAJE_TNOH, bins=60, color=PALETA[2], alpha=.85)
ax.axvline(P75, color=PALETA[0], ls='--', lw=2, label=f'p75 = {P75:.1f}%  →  corte del target')
ax.set_xlabel('TNOH (%)'); ax.set_title('Distribución del target'); ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# ---------- CALENDARIO ----------
df['MES_SIN'] = np.sin(2*np.pi*df.MES/12)        # ciclicidad: diciembre y enero quedan cerca
df['MES_COS'] = np.cos(2*np.pi*df.MES/12)
df['TRIMESTRE']          = ((df.MES - 1)//3) + 1
df['ES_FIESTAS_PATRIAS'] = df.MES.eq(7).astype(int)
df['ES_SEMANA_SANTA']    = df.MES.isin([3,4]).astype(int)
df['ES_FIN_DE_ANIO']     = df.MES.isin([12,1]).astype(int)
df['ES_TEMPORADA_SECA']  = df.MES.isin([5,6,7,8,9]).astype(int)     # clave en la sierra
df['ES_COVID']           = ((df.FECHA >= '2020-03-01') & (df.FECHA <= '2021-12-01')).astype(int)
df['ES_POST_COVID']      = (df.FECHA > '2021-12-01').astype(int)
df['T']                  = (df.FECHA.dt.year - 2015)*12 + df.FECHA.dt.month   # tendencia

# ---------- REZAGOS (por perfil: mismo depto + clase + categoría) ----------
PERFIL = ['DEPARTAMENTO','CLASE','CATEGORIA']
g = df.groupby(PERFIL)['PORCENTAJE_TNOH']
df['TNOH_LAG1']  = g.shift(1)
df['TNOH_LAG3']  = g.shift(3)
df['TNOH_LAG12'] = g.shift(12)                                        # mismo mes del año pasado
df['TNOH_MM3']   = g.transform(lambda s: s.shift(1).rolling(3).mean())
df['TNOH_MM12']  = g.transform(lambda s: s.shift(1).rolling(12).mean())
df['DELTA_YOY']  = df.TNOH_LAG1 - df.TNOH_LAG12

# ---------- ESTRUCTURA ----------
df['HAB_POR_ESTAB']  = df.NUMERO_HABITACIONES / df.NUMERO_ESTABLECIMIENTOS.replace(0, np.nan)
df['PLAZAS_POR_HAB'] = df.NUMERO_PLAZAS_CAMA  / df.NUMERO_HABITACIONES.replace(0, np.nan)
df['LOG_ESTAB']      = np.log1p(df.NUMERO_ESTABLECIMIENTOS)   # log: evita que Lima domine la escala
df['LOG_HAB']        = np.log1p(df.NUMERO_HABITACIONES)
df['N_ESTRELLAS']    = df.CATEGORIA.map({'1 ESTRELLA':1,'2 ESTRELLAS':2,'3 ESTRELLAS':3,
                                         '4 ESTRELLAS':4,'5 ESTRELLAS':5}).fillna(0).astype(int)
print("Calendario, rezagos y estructura: OK")

In [ ]:
# ---------- PERFIL HISTÓRICO DEL DEPARTAMENTO (anti-leakage: solo 2015-2019) ----------
base = limpio[limpio.ANIO <= 2019]
perfil_dep = base.groupby('DEPARTAMENTO').apply(lambda x: pd.Series({
    'DEP_PCT_EXT_HIST'  : x.TOTAL_ARRIBOS_EXT.sum() / max(x.TOTAL_ARRIBOS.sum(), 1) * 100,
    'DEP_TNOH_HIST'     : x.PORCENTAJE_TNOH.mean(),
    'DEP_ESTACIONALIDAD': x.groupby('MES').PORCENTAJE_TNOH.mean().std(),
}), include_groups=False)

df = df.merge(perfil_dep, on='DEPARTAMENTO', how='left')
print(perfil_dep.sort_values('DEP_PCT_EXT_HIST', ascending=False).round(1).head(8).to_string())
print("\n>>> El % de extranjeros varía muchísimo entre departamentos (de ~0.5% a ~60%).")
print(">>> Ese rango amplio es lo que permite al modelo aprender la relación.")

In [ ]:
# Los rezagos exigen 12 meses de historia por perfil
antes = len(df)
df = df.dropna(subset=['TNOH_LAG12','TNOH_MM12']).copy()
print(f"Filas tras exigir 12 meses de historia: {antes:,} → {len(df):,}")

FEATURES = [
 # calendario
 'MES','MES_SIN','MES_COS','TRIMESTRE','ES_FIESTAS_PATRIAS','ES_SEMANA_SANTA',
 'ES_FIN_DE_ANIO','ES_TEMPORADA_SECA','ES_COVID','ES_POST_COVID','T','ANIO',
 # rezagos
 'TNOH_LAG1','TNOH_LAG3','TNOH_LAG12','TNOH_MM3','TNOH_MM12','DELTA_YOY',
 # estructura
 'NUMERO_ESTABLECIMIENTOS','NUMERO_HABITACIONES','HAB_POR_ESTAB','PLAZAS_POR_HAB',
 'LOG_ESTAB','LOG_HAB','N_ESTRELLAS',
 # perfil histórico del departamento
 'DEP_PCT_EXT_HIST','DEP_TNOH_HIST','DEP_ESTACIONALIDAD',
 # categóricas (one-hot / target encoding en el modelado)
 'DEPARTAMENTO','CLASE','CATEGORIA',
]
print(f"\nFEATURES: {len(FEATURES)}   |   EXCLUIDAS por leakage: {len(LEAKAGE)}")

---
## Etapa 6 — Split temporal y exportación

**Split temporal, NO aleatorio.** Un `train_test_split` aleatorio pondría meses de 2025 en el train y meses de 2016 en el test: el modelo vería el futuro. **Eso también es leakage**.

| Conjunto | Periodo |
|---|---|
| **Train** | 2016 – 2023 |
| **Test** | 2024 – 2025-S1 |

In [ ]:
train = df[df.ANIO <= 2023]
test  = df[df.ANIO >= 2024]

print(f"TRAIN 2016–2023 : {len(train):,} filas   |  clase positiva: {train.Y_ALTA_OCUPACION.mean()*100:.1f}%")
print(f"TEST  2024–2025 : {len(test):,} filas   |  clase positiva: {test.Y_ALTA_OCUPACION.mean()*100:.1f}%")
print("\n>>> Desbalance ~75/25: moderado. No hace falta SMOTE; en el modelado se usa class_weight='balanced'.")

In [ ]:
# ---------- EXPORTACIÓN ----------
analitico = df[FEATURES + ['Y_ALTA_OCUPACION','PORCENTAJE_TNOH','FECHA','NO_REPORTO']]
analitico.to_csv('ocupaperu_analitico.csv', index=False)
print(f"✓ ocupaperu_analitico.csv   {analitico.shape[0]:,} filas x {analitico.shape[1]} cols    (clasificación y regresión)")

# Serie mensual nacional — usa 'limpio' para conservar los 126 meses
serie = limpio.groupby('FECHA').apply(lambda x: pd.Series({
    'TNOH_NACIONAL' : np.average(x.PORCENTAJE_TNOH, weights=x.NUMERO_HABITACIONES),
    'ARRIBOS'       : x.TOTAL_ARRIBOS.sum(),
    'ARRIBOS_NAC'   : x.TOTAL_ARRIBOS_NAC.sum(),
    'ARRIBOS_EXT'   : x.TOTAL_ARRIBOS_EXT.sum(),
}), include_groups=False).reset_index()
serie.to_csv('ocupaperu_serie.csv', index=False)
print(f"✓ ocupaperu_serie.csv       {len(serie)} meses x {serie.shape[1]-1} series          (pronóstico)")

# Descarga a tu computador (Colab)
from google.colab import files
for f in ['ocupaperu_analitico.csv', 'ocupaperu_serie.csv']:
    files.download(f)
print("\nArchivos guardados en /content y descargados. Súbelos a la Fase 4 (también en /content).")


---
## Resumen de la Fase 3

| | Filas |
|---|---|
| Crudas (11 archivos) | **61,890** |
| − agregados `TODAS CONSOLIDADAS` | −33,737 |
| − imposibles y no reportados | −1,610 |
| − sin 12 meses de historia (rezagos) | −2,841 |
| **= Dataset analítico** | **≈ 23,700** |
| Features | **31** |
| **Excluidas por leakage** | **11** |

### Puntos clave

1. **Schema drift** — dos esquemas incompatibles; `concat()` sin normalizar falla **en silencio**.
2. **Agregados** — las filas `TODAS CONSOLIDADAS` contarían cada hotel 2–3 veces.
3. **Leakage** — `PORCENTAJE_TNOC` (r ≈ 0.93) y `TOTAL_PERNOCT` reconstruyen el target. **Fuera.**
4. **Split temporal** — aleatorio = filtrar el futuro al pasado. También es leakage.
5. **Sesgo de supervivencia** — el TNOH no cayó a 0 en el COVID porque los hoteles cerrados salen del denominador. Los arribos sí: **−91%**.

> **Al quitar las variables con fuga, el ROC-AUC baja de ~0.99 a ~0.95.**
> **Y ese es exactamente el punto: el 0.99 era engañoso. El 0.95 se puede usar.**

---
**Siguiente:** Fase 4 — Modelado (regresión, clasificación, SHAP y pronóstico).